# 交叉熵损失函数

上一章我们发现：在输出层后加入 Softmax 激活函数后，准确率反而下降了。其原因是 **Softmax 激活函数与均方误差损失函数搭配不当**。

我们来仔细看看问题出在哪里。均方误差衡量的是预测值与标签值的数值差距：误差越大，惩罚越大，梯度也越大。但 Softmax 的输出是概率，当预测非常错误时（比如对正确类别的输出概率只有 0.001），Softmax 曲线已经进入了饱和区——输出几乎不再随输入变化，梯度接近于零。

这就造成了一个逻辑矛盾：**预测越错，修正越小。** MSE 认为误差很大、应该大力修正，但 Softmax 饱和区的梯度把这个修正信号几乎全部消掉了。两者合起来，训练变得极其缓慢。

---

解决方案是换一种损失函数，让它能与 Softmax 激活函数的饱和特性相互抵消。这就是**交叉熵损失函数**（Cross-Entropy Loss）。

交叉熵来自信息论，衡量的是「预测的概率分布」与「真实分布」之间的差距。它的核心特性恰好与 Softmax 激活函数的饱和互补：
- 预测概率越接近真实标签，损失越小、梯度也越小（收敛时不过度修正）
- 预测概率越偏离真实标签，损失急剧增大，产生强烈的修正信号

更重要的是，当把 Softmax 激活函数与交叉熵损失函数的**梯度公式**合并推导时，Softmax 损失函数的**指数运算**和交叉熵损失函数的**对数运算**正好相互抵消，得到一个极其简洁的梯度：$p_i - y_i$（预测概率减去真实标签）。

``💡 这也是为什么实现 CELoss 时，我们通常把 Softmax 计算也包含在内。``

In [1]:
from abc import abstractmethod, ABC

import numpy as np

np.random.seed(99)

## 基础架构

### 张量

In [2]:
class Tensor:

    def __init__(self, data):
        self.data = np.array(data)
        self.grad = np.zeros_like(self.data)
        self.gradient_fn = None
        self.parents = set()

    def backward(self):
        if self.gradient_fn is not None:
            self.gradient_fn()

        for p in self.parents:
            p.backward()

    @property
    def shape(self):
        return self.data.shape

    def __repr__(self):
        return f'Tensor({self.data})'

### 基础数据集

In [3]:
class Dataset(ABC):

    def __init__(self, batch_size=1):
        self.batch_size = batch_size

        self.test_features = None
        self.test_labels = None
        self.train_features = None
        self.train_labels = None

        self.load()

        self.features = self.train_features
        self.labels = self.train_labels

    @abstractmethod
    def load(self):
        pass

    def train(self):
        self.features = self.train_features
        self.labels = self.train_labels

    def eval(self):
        self.features = self.test_features
        self.labels = self.test_labels

    def items(self):
        return Tensor(self.features), Tensor(self.labels)

    def __len__(self):
        return len(self.features) // self.batch_size

    def __getitem__(self, index):
        start = index * self.batch_size
        end = start + self.batch_size

        feature = Tensor(self.features[start: end])
        label = Tensor(self.labels[start: end])
        return feature, label

### 基础层

In [4]:
class Layer(ABC):

    def __init__(self):
        self.training = True

    def __call__(self, x: Tensor):
        return self.forward(x)

    def train(self):
        self.training = True

    def eval(self):
        self.training = False

    @abstractmethod
    def forward(self, x: Tensor):
        pass

    @property
    def parameters(self):
        return []

    def __repr__(self):
        return f"{type(self).__name__}[]"

### 基础损失函数

In [5]:
class Loss(ABC):

    def __call__(self, p: Tensor, y: Tensor):
        return self.loss(p, y)

    @abstractmethod
    def loss(self, p: Tensor, y: Tensor):
        pass

### 基础优化器

In [6]:
class Optimizer(ABC):

    def __init__(self, parameters, lr):
        self.parameters = parameters
        self.lr = lr

    def reset(self):
        for p in self.parameters:
            p.grad = np.zeros_like(p.data)

    @abstractmethod
    def step(self):
        pass

### 基础模型

In [7]:
class Model(ABC):

    def __init__(self, layer, loss_fn, optimizer):
        self.layer = layer
        self.loss_fn = loss_fn
        self.optimizer = optimizer

    @abstractmethod
    def train(self, dataset, epochs):
        pass

    @abstractmethod
    def test(self, dataset):
        pass

## 数据

### MNIST 数据集

In [8]:
class MNISTDataset(Dataset):

    def __init__(self, filename, batch_size=1):
        self.filename = filename
        super().__init__(batch_size)

    def load(self):
        with np.load(self.filename, allow_pickle=True) as f:
            self.train_features, self.train_labels = self.preprocess(f['x_train'], f['y_train'])
            self.test_features, self.test_labels = self.preprocess(f['x_test'], f['y_test'])

    @staticmethod
    def preprocess(x, y):
        inputs = x / 255.0
        inputs = np.expand_dims(inputs, axis=1)
        targets = np.zeros((len(y), 10))
        targets[np.arange(len(y)), y] = 1
        return inputs, targets

    def estimate(self, predictions):
        predicted_digits = predictions.data.argmax(axis=1)
        digits = self.labels.argmax(axis=1)
        correct = (predicted_digits == digits).sum()
        return correct / len(self.labels)

## 模型

### 线性层

In [9]:
class Linear(Layer):

    def __init__(self, in_size, out_size):
        super().__init__()
        self.weight = Tensor(np.random.rand(out_size, in_size) / in_size)
        self.bias = Tensor(np.random.rand(out_size))

    def forward(self, x: Tensor):
        p = Tensor(x.data @ self.weight.data.T + self.bias.data)

        def gradient_fn():
            self.weight.grad += p.grad.T @ x.data
            self.bias.grad += np.sum(p.grad, axis=0)
            x.grad += p.grad @ self.weight.data

        p.gradient_fn = gradient_fn
        p.parents = {x}
        return p

    @property
    def parameters(self):
        return [self.weight, self.bias]

    def __repr__(self):
        return f'Linear[weight{self.weight.shape}; bias{self.bias.shape}]'

### 顺序层

In [10]:
class Sequential(Layer):

    def __init__(self, layers):
        super().__init__()
        self.layers = layers

    def train(self):
        super().train()
        for l in self.layers:
            l.train()

    def eval(self):
        super().eval()
        for l in self.layers:
            l.eval()

    def forward(self, x: Tensor):
        for l in self.layers:
            x = l(x)
        return x

    @property
    def parameters(self):
        return [p for l in self.layers for p in l.parameters]

    def __repr__(self):
        return '\n'.join(str(l) for l in self.layers if str(l))

### 展平层

In [11]:
class Flatten(Layer):

    def forward(self, x: Tensor):
        p = Tensor(np.array(x.data.reshape(x.data.shape[0], -1)))

        def gradient_fn():
            x.grad += p.grad.reshape(x.data.shape)

        p.gradient_fn = gradient_fn
        p.parents = {x}
        return p

### 丢弃层

In [12]:
class Dropout(Layer):

    def __init__(self, dropout_rate=0.2):
        super().__init__()
        self.dropout_rate = dropout_rate

    def forward(self, x: Tensor):
        if not self.training:
            return x

        mask = np.random.random(x.data.shape) > self.dropout_rate
        p = Tensor(x.data * mask)

        def gradient_fn():
            x.grad += p.grad * mask

        p.gradient_fn = gradient_fn
        p.parents = {x}
        return p

    def __repr__(self):
        return f'Dropout[rate={self.dropout_rate}]'

### ReLU 激活函数层

In [13]:
class ReLU(Layer):

    def forward(self, x: Tensor):
        a = Tensor(np.maximum(0, x.data))

        def gradient_fn():
            x.grad += a.grad * (a.data > 0)

        a.gradient_fn = gradient_fn
        a.parents = {x}
        return a

### Tanh 激活函数层

In [14]:
class Tanh(Layer):

    def forward(self, x: Tensor):
        a = Tensor(np.tanh(x.data))

        def gradient_fn():
            x.grad += a.grad * (1 - a.data ** 2)

        a.gradient_fn = gradient_fn
        a.parents = {x}
        return a

### Sigmoid 激活函数层

In [15]:
class Sigmoid(Layer):

    def __init__(self, clip_range=(-100, 100)):
        super().__init__()
        self.clip_range = clip_range

    def forward(self, x: Tensor):
        z = np.clip(x.data, self.clip_range[0], self.clip_range[1])
        a = Tensor(1 / (1 + np.exp(-z)))

        def gradient_fn():
            x.grad += a.grad * a.data * (1 - a.data)

        a.gradient_fn = gradient_fn
        a.parents = {x}
        return a

### Softmax 激活函数层

In [16]:
class Softmax(Layer):

    def __init__(self, axis=-1):
        super().__init__()
        self.axis = axis

    def forward(self, x: Tensor):
        exp = np.exp(x.data - np.max(x.data, axis=self.axis, keepdims=True))
        a = Tensor(exp / np.sum(exp, axis=self.axis, keepdims=True))

        def gradient_fn():
            grad = np.sum(a.data * a.grad, axis=self.axis, keepdims=True)
            x.grad += a.data * (a.grad - grad)

        a.gradient_fn = gradient_fn
        a.parents = {x}
        return a

### 均方误差损失函数

In [17]:
class MSELoss(Loss):

    def loss(self, p: Tensor, y: Tensor):
        mse = Tensor(np.mean(np.square(y.data - p.data)))

        def gradient_fn():
            p.grad += -2 * (y.data - p.data) / len(y.data)

        mse.gradient_fn = gradient_fn
        mse.parents = {p}
        return mse

### 交叉熵损失函数

**前向传播**：给定一批预测值 $p$（网络输出，尚未经过 Softmax 激活函数）和单热标签 $y$，交叉熵损失定义为：
$$
L = -\frac{1}{N}\sum_{n=1}^{N} \sum_{i=1}^{C} y_{n,i} \log(a_{n,i})
$$

其中 $a_{n,i} = \text{Softmax}(p_{n,i})$ 是经过 Softmax 后第 $n$ 个样本属于第 $i$ 类的概率，$N$ 是批大小，$C$ 是类别数。

由于 $y$ 是单热编码（只有正确类别为 1，其余为 0），求和只保留正确类别那一项：
$$
L = -\frac{1}{N}\sum_{n=1}^{N} \log(a_{n, k_n})
$$
其中 $k_n$ 是第 $n$ 个样本的正确类别。**损失就是正确类别的预测概率取对数后取反**：概率越高损失越小，概率越低损失越大（且随概率趋近于 0 而急剧增大）。

**梯度计算**：将 Softmax 和交叉熵合并推导（Softmax 的指数与 log 相消），得到极其简洁的梯度：
$$
\frac{\partial L}{\partial p_i} = \frac{1}{N}(a_i - y_i)
$$
即：预测概率减去真实标签，正确类别的梯度是负的（促使其概率增大），其余类别的梯度是正的（促使其概率减小）。这个梯度公式和 MSE 一样直观，但背后已经隐含了 Softmax 的计算。

``💡 CELoss 损失函数内部已经计算了 Softmax，因此使用 CELoss 损失函数时，网络的输出层不需要再加 Softmax 激活函数。否则会导致 Softmax 被使用两次，结果出错。``

In [18]:
class CELoss(Loss):

    def loss(self, p: Tensor, y: Tensor):
        exp = np.exp(p.data - np.max(p.data, axis=-1, keepdims=True))
        softmax = exp / np.sum(exp, axis=-1, keepdims=True)

        log = np.log(np.clip(softmax, 1e-10, 1))
        ce = Tensor(0 - np.sum(y.data * log) / len(y.data))

        def gradient_fn():
            p.grad += (softmax - y.data) / len(y.data)

        ce.gradient_fn = gradient_fn
        ce.parents = {p}
        return ce

### 二元交叉熵损失函数

交叉熵损失适用于多分类（$C > 2$ 个类别）。对于**二分类**问题，输出层只有 1 个神经元，配合 Sigmoid 激活函数输出一个概率值 $a \in (0, 1)$，此时应使用**二元交叉熵**（Binary Cross-Entropy，BCE）。

**前向传播**：
$$
L = -\frac{1}{N}\sum_{n=1}^{N}\left[y_n \log(a_n) + (1 - y_n) \log(1 - a_n)\right]
$$

直觉上：当真实标签 $y=1$ 时，只有 $-\log(a)$ 项有效：$a$ 越接近 1 损失越小；当 $y=0$ 时，只有 $-\log(1-a)$ 项有效：$a$ 越接近 0 损失越小。

**梯度计算**：对 Sigmoid 输出 $a$ 求导（注意：这里直接对激活值求导，不像 CELoss 那样内置 Sigmoid）：
$$
\frac{\partial L}{\partial a} = \frac{1}{N}\cdot\frac{a - y}{a(1 - a)}
$$

BCE 的梯度分母 $a(1-a)$ 与 Sigmoid 的导数 $a(1-a)$ 相同，两者相乘时正好约分，消除了 Sigmoid 在两端的梯度饱和效应，使训练信号保持稳定。

``💡 BCELoss 损失函数接收的是 Sigmoid 激活函数之后的激活值，需要先在网络中显式添加 Sigmoid 激活函数层。这与 CELoss 损失函数内置 Softmax 不同。``

In [19]:
class BCELoss(Loss):

    def loss(self, p: Tensor, y: Tensor):
        clipped = np.clip(p.data, 1e-7, 1 - 1e-7)
        bce = Tensor(-np.mean(y.data * np.log(clipped) + (1 - y.data) * np.log(1 - clipped)))

        def gradient_fn():
            p.grad += (clipped - y.data) / (clipped * (1 - clipped)) / len(y.data)

        bce.gradient_fn = gradient_fn
        bce.parents = {p}
        return bce

### 随机梯度下降优化器

In [20]:
class SGDOptimizer(Optimizer):

    def step(self):
        for p in self.parameters:
            p.data -= p.grad * self.lr

### 神经网络模型

In [21]:
class NNModel(Model):

    def train(self, dataset, epochs):
        self.layer.train()
        dataset.train()

        for epoch in range(epochs):
            for i in range(len(dataset)):
                features, labels = dataset[i]

                self.optimizer.reset()
                predictions = self.layer(features)
                loss = self.loss_fn(predictions, labels)
                loss.backward()
                self.optimizer.step()

    def test(self, dataset):
        self.layer.eval()
        dataset.eval()

        features, labels = dataset.items()
        predictions = self.layer(features)
        loss = self.loss_fn(predictions, labels)
        return predictions, loss

## 训练

### 超参数：学习率

In [22]:
LEARNING_RATE = 0.01

### 超参数：批大小

In [23]:
BATCH_SIZE = 2

### 超参数：轮次

In [24]:
EPOCHS = 10

### 建模

使用 CELoss 损失函数时，输出层后面**不需要** Softmax 激活函数。CELoss 损失函数内部已经包含了 Softmax 的计算。

In [25]:
dataset = MNISTDataset('tinymnist.npz', BATCH_SIZE)

layer = Sequential([Flatten(),
                    Linear(1 * 28 * 28, 64),
                    ReLU(),
                    Dropout(),
                    Linear(64, 10)])
loss_fn = CELoss()
optimizer = SGDOptimizer(layer.parameters, lr=LEARNING_RATE)

model = NNModel(layer, loss_fn, optimizer)
print(layer)

Flatten[]
Linear[weight(64, 784); bias(64,)]
ReLU[]
Dropout[rate=0.2]
Linear[weight(10, 64); bias(10,)]


### 迭代

In [26]:
model.train(dataset, EPOCHS)

## 验证

### 测试

In [27]:
predictions, loss = model.test(dataset)
accuracy = dataset.estimate(predictions)
print(f'accuracy: {accuracy:.2%}')

accuracy: 89.80%


换用 CELoss 损失函数后，准确率从约 87.3% 提升到 89.8%。

``💡 在完整 MNIST（6 万张训练图）上，CELoss 的优势会更加明显。``

## 课后练习

分别用 **MSELoss**（无 Softmax）和 **CELoss**（无 Softmax）训练模型各 10 轮，比较准确率。再把轮次增加到 30 轮、50 轮、100 轮，各自的变化有什么特点？